In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [2]:
import os
import shutil
import torch
import re
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

# SETUP & CONFIGURATION
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

try:
    import huggingface_hub.utils._progress
    import tqdm.std
    huggingface_hub.utils._progress.tqdm = tqdm.std.tqdm
except Exception:
    pass

RAW_IMAGES_FOLDER = "./03_RAW_IMAGES/"       
CLEAN_IMAGE_FOLDER = "./03_CLEAN_IMAGES/"   

os.makedirs(RAW_IMAGES_FOLDER, exist_ok=True)
os.makedirs(CLEAN_IMAGE_FOLDER, exist_ok=True)

print("Loading CLIP Model OFFLINE from local folder...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Hardware Acceleration: {device.upper()}")

model_id = "./local_clip_model" 
model = CLIPModel.from_pretrained(model_id).to(device)
processor = CLIPProcessor.from_pretrained(model_id)

CATEGORIES = [
    "a clear photograph focusing on a single house exterior or a specific room inside a house", 
    "a wide street view, cityscape skyline, or village landscape showing many different buildings", 
    "a portrait or photograph focusing on people, such as a couple walking or a man posing", 
    "an extreme close-up of hands working, brick textures, or specific objects without showing the whole structure", 
    "a graphic design logo, icon, scanned document, text, or blank page" 
]

VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')

# Memory bank to store the mathematical embeddings of saved images
seen_embeddings = None

# MAIN AI FILTER PIPELINE
def process_images():
    global seen_embeddings
    print("\nStarting Image Filtering & Deduplication...")
    total_processed = 0
    total_saved = 0
    
    for image_name in os.listdir(RAW_IMAGES_FOLDER):
        
        if not image_name.lower().endswith(VALID_EXTENSIONS):
            continue
            
        total_processed += 1
        
        # STAGE 1: Fast Name Filtering 
        # Looking for patterns like "(1)", "(2)", "(copy)" in the filename
        if re.search(r'\(\d+\)', image_name):
            print(f"  [-] Dropped (Name Duplicate): {image_name}")
            continue

        image_path = os.path.join(RAW_IMAGES_FOLDER, image_name)
        
        try:
            image = Image.open(image_path).convert("RGB")
            inputs = processor(text=CATEGORIES, images=image, return_tensors="pt", padding=True).to(device)
            
            with torch.no_grad():
                outputs = model(**inputs)
                
                # STAGE 2: Visual Embedding Deduplication
                # Extracting the visual math vector and normalize it
                image_embeds = outputs.image_embeds
                image_embeds = image_embeds / image_embeds.norm(p=2, dim=-1, keepdim=True)
                
                # If we have saved images, compare the current image to the memory bank
                if seen_embeddings is not None:
                    # Calculate Cosine Similarity (1.0 = identical, 0.0 = completely different)
                    similarities = torch.matmul(image_embeds, seen_embeddings.T)
                    
                    # If it is 98% visually similar to an image we already saved, drop it!
                    if similarities.max().item() > 0.98:
                        print(f"  [-] Dropped (Visual Duplicate): {image_name}")
                        continue
                
                # Getting the standard classification probabilities
                probs = outputs.logits_per_image.softmax(dim=-1)
            
            prob_house = probs[0][0].item()
            prob_landscape = probs[0][1].item()
            prob_person = probs[0][2].item()
            prob_closeup = probs[0][3].item()
            prob_document = probs[0][4].item()
            
            max_prob = max(prob_house, prob_landscape, prob_person, prob_closeup, prob_document)
            
            # RULE: It must predict "Single House" as the #1 category with > 55% confidence
            if max_prob == prob_house and prob_house > 0.55:
                save_path = os.path.join(CLEAN_IMAGE_FOLDER, image_name)
                shutil.copy(image_path, save_path)
                total_saved += 1
                
                # IMPORTANT: Save this approved image's embedding into our memory bank!
                if seen_embeddings is None:
                    seen_embeddings = image_embeds
                else:
                    seen_embeddings = torch.cat((seen_embeddings, image_embeds), dim=0)
                    
                print(f"  [+] Kept (House): {image_name} (Confidence: {prob_house:.1%})")
            else:
                if max_prob == prob_landscape:
                    reason = "Cityscape / Street View Detected"
                elif max_prob == prob_person:
                    reason = "Person / Portrait Detected"
                elif max_prob == prob_closeup:
                    reason = "Extreme Close-up / Action Detected"
                elif max_prob == prob_document:
                    reason = "Document / Logo Detected"
                else:
                    reason = "Low Confidence / Ambiguous"
                    
                print(f"  [-] Dropped ({reason}): {image_name}")
                    
        except Exception as e:
            print(f"Error reading {image_name}: {e}")

    print("\n" + "="*40)
    print("PIPELINE COMPLETE")
    print(f"Total raw images processed: {total_processed}")
    print(f"Total valid, unique photos saved: {total_saved}")
    print(f"Removed {total_processed - total_saved} useless or duplicate images!")
    print("="*40)

if __name__ == "__main__":
    process_images()

Loading CLIP Model OFFLINE from local folder...
Hardware Acceleration: CPU


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


Starting Image Filtering & Deduplication...
  [-] Dropped (Person / Portrait Detected): 82d37049-c96d-491c-ab92-9d9d75ee1482.jpg
  [+] Kept (House): buildings-or-apartment-under-construction-with-storm-clouds-.jpg (Confidence: 86.9%)
  [-] Dropped (Extreme Close-up / Action Detected): construction.jpg
  [-] Dropped (Cityscape / Street View Detected): delhi-cityscape.jpg
  [-] Dropped (Person / Portrait Detected): freddy-211.jpeg
  [-] Dropped (Name Duplicate): free-photo-of-a-black-and-white-photo-of-a-lighthouse (6).jpeg
  [-] Dropped (Name Duplicate): free-photo-of-a-blue-building-with-a-blue-roof-and-palm-tree (1).jpeg
  [-] Dropped (Name Duplicate): free-photo-of-a-blue-building-with-a-blue-roof-and-palm-tree (2).jpeg
  [-] Dropped (Name Duplicate): free-photo-of-a-blue-building-with-a-blue-roof-and-palm-tree (3).jpeg
  [-] Dropped (Name Duplicate): free-photo-of-a-blue-building-with-a-blue-roof-and-palm-tree (4).jpeg
  [-] Dropped (Name Duplicate): free-photo-of-a-blue-building-w

C:\Users\Harsh Datt\anaconda3\Lib\site-packages\PIL\Image.py:3432: DecompressionBombWarning: Image size (159120000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


  [-] Dropped (Person / Portrait Detected): pexels-photo-17935458.jpeg
  [+] Kept (House): pexels-photo-17939435.jpeg (Confidence: 57.9%)
  [+] Kept (House): pexels-photo-18025799.jpeg (Confidence: 95.4%)
  [+] Kept (House): pexels-photo-18048181.jpeg (Confidence: 91.6%)
  [-] Dropped (Cityscape / Street View Detected): pexels-photo-18132023.jpeg
  [+] Kept (House): pexels-photo-18205631.jpeg (Confidence: 77.5%)
  [-] Dropped (Person / Portrait Detected): pexels-photo-18276994.jpeg
  [-] Dropped (Person / Portrait Detected): pexels-photo-18276996.jpeg
  [+] Kept (House): pexels-photo-18364647.jpeg (Confidence: 99.1%)
  [-] Dropped (Visual Duplicate): pexels-photo-18408538.jpeg
  [+] Kept (House): pexels-photo-18529267.jpeg (Confidence: 99.1%)
  [-] Dropped (Name Duplicate): pexels-photo-1862402 (1).jpeg
  [-] Dropped (Name Duplicate): pexels-photo-1862402 (2).jpeg
  [+] Kept (House): pexels-photo-1862402.jpeg (Confidence: 99.9%)
  [+] Kept (House): pexels-photo-18712316.jpeg (Confidenc